Dataset: a plain-text corpus — e.g. a public-domain book from Project Gutenberg (something like a Sherlock Holmes story — enough text to get real bigram statistics, not too huge to slow you down).

Load text, tokenize into words (simple .split()
lowercase + strip punctuation to start).





Build unigram counts and bigram counts (dictionaries or Counter).





Add <s> / </s> sentence boundary tokens (split corpus into sentences first — naive split on ./!/? is fine).






Implement bigram_prob(w2, w1) using count ratios (Def 8.2.1 / eq 8.13).





Add Laplace (add-1) smoothing (eq 8.19/8.20) — compare a few sentence probabilities with vs. without smoothing, including one with a bigram that never appears in training (should go from 0 to nonzero).





Implement a text generator: start at <s>, sample next word from the bigram distribution, repeat until </s>. Generate a few sample sentences.





Implement perplexity (Def 8.4.4) on a held-out test split of the corpus (e.g. last 10% of sentences).

In [6]:
import numpy as np
import urllib.request
url = "https://www.gutenberg.org/files/2701/2701-0.txt"
urllib.request.urlretrieve(url, "moby_dick.txt")
with open("moby_dick.txt", "r", encoding="utf-8") as f:
    text_full = f.read()
#tokenize text into words
start = text_full.find("Call me Ishmael. Some years ago")
end = text_full.find("it rolled five thousand years ago.")
text_t = text_full[start:end]
text = text_t.lower().split()
sentence_current = ['<s>']
text_sentences = []
for word in text:
  if "." in word or "!" in word or "?" in word:
    word = word.strip('.!?,:;“”')
    sentence_current.append(word)
    sentence_current.append('</s>')
    text_sentences.append(sentence_current)
    sentence_current = ['<s>']
  else:
    word = word.strip(',:;“”')
    sentence_current.append(word)
print(len(text_sentences))
print(text_sentences[:3])
#unigram and bigram counts
unigram_count= {}
for sentence in text_sentences:
  for word in sentence:
    unigram_count[word] = (unigram_count.get(word, 0)+1)
bigram_count = {}
for sentence in text_sentences:
  for i in range(len(sentence)-1):
    bigram_count[(sentence[i], sentence[i+1])] = (bigram_count.get((sentence[i], sentence[i+1]), 0))+1
print(len(unigram_count))
print(len(bigram_count))
print(unigram_count['<s>'])
print(unigram_count['</s>'])
#bigram probabilities
def bigram_prob(w2, w1):
  if unigram_count.get(w1, 0) != 0:
    return bigram_count.get((w1, w2), 0)/unigram_count.get(w1)
  else:
    return 0
bigram_prob('find', 'i')
#laplace smoothing
def bigram_prob_smoothed(w2, w1):
  return (bigram_count.get((w1, w2), 0)+1)/(unigram_count.get(w1, 0)+len(unigram_count))
bigram_prob_smoothed('find', 'i')
#text generator
current_word = '<s>'
generated_sentence = []
while current_word != '</s>' and len(generated_sentence)<50:
  candidates = []
  prob = []
  generated_sentence.append(current_word)
  for (w1, w2) in bigram_count.keys():
    if w1 == current_word:
      candidates.append(w2)
      prob.append(bigram_prob(w2, w1))
  current_word = np.random.choice(candidates, p=prob)
generated_sentence.append('</s>')
print(generated_sentence)
#perplexity, PP(W)=exp(−1/N​i∑​logP(wi​∣wi−1​))
test_amount = int(len(text_sentences)*0.9)
train_sentences = text_sentences[ :test_amount]
test_sentences = text_sentences[test_amount: ]
unigram_count_train= {}
for sentence in train_sentences:
  for word in sentence:
    unigram_count_train[word] = (unigram_count_train.get(word, 0)+1)
bigram_count_train = {}
for sentence in train_sentences:
  for i in range(len(sentence)-1):
    bigram_count_train[(sentence[i], sentence[i+1])] = (bigram_count_train.get((sentence[i], sentence[i+1]), 0))+1
def bigram_train_prob_smoothed(w2, w1):
  return (bigram_count_train.get((w1, w2), 0)+1)/(unigram_count_train.get(w1, 0)+len(unigram_count_train))
log_prob_sum = 0
bigram_sum = 0
for sentence in test_sentences:
  for i in range(len(sentence)-1):
    log_prob_sum += np.log((bigram_train_prob_smoothed(sentence[i+1], sentence[i])))
    bigram_sum +=1
perplexity = np.exp((-1/bigram_sum)*(log_prob_sum))
print(f"perplexity = {perplexity:.4f}")

10055
[['<s>', 'call', 'me', 'ishmael', '</s>'], ['<s>', 'some', 'years', 'ago—never', 'mind', 'how', 'long', 'precisely—having', 'little', 'or', 'no', 'money', 'in', 'my', 'purse', 'and', 'nothing', 'particular', 'to', 'interest', 'me', 'on', 'shore', 'i', 'thought', 'i', 'would', 'sail', 'about', 'a', 'little', 'and', 'see', 'the', 'watery', 'part', 'of', 'the', 'world', '</s>'], ['<s>', 'it', 'is', 'a', 'way', 'i', 'have', 'of', 'driving', 'off', 'the', 'spleen', 'and', 'regulating', 'the', 'circulation', '</s>']]
20587
114768
10055
10055
['<s>', np.str_('by'), np.str_('the'), np.str_('landlady'), np.str_('should'), np.str_('intervene'), np.str_('and'), np.str_('with'), np.str_('that'), np.str_('his'), np.str_('crow’s-nest'), np.str_('of'), np.str_('a'), np.str_('most'), np.str_('unaccountable'), np.str_('elijah'), np.str_('hailing'), np.str_('the'), np.str_('hardy'), np.str_('peasants'), np.str_('of'), np.str_('nantucket'), np.str_('ship'), np.str_('will'), np.str_('consume'), np.s